In [19]:
# Install Ollama (run once)
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
76 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as re

In [20]:
import subprocess, time, requests, os, signal

# Lance le serveur (en arrière-plan)
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Attendre que l'API réponde
for _ in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=1)
        if r.status_code == 200:
            print("Ollama API up")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama API ne répond pas (serve n'a pas démarré)")


✅ Ollama API up


In [21]:
# Download the model
!ollama pull llama3.2

In [22]:
!pip install ollama requests

In [23]:
import json
import re
import requests
from typing import Dict, List, Tuple
from dataclasses import dataclass

try:
    import ollama
    OLLAMA_PACKAGE_AVAILABLE = True
    print("ollama package available")
except ImportError:
    OLLAMA_PACKAGE_AVAILABLE = False
    print("ollama package not available, using requests fallback")

ollama package available


In [24]:
@dataclass
class GraphSchema:
    """RDF graph schema matching the actual RML mapping."""

    prefixes = """
PREFIX schema: <http://schema.org/>
PREFIX sosa:   <http://www.w3.org/ns/sosa/>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
"""

    schema_description = """
Knowledge graph about cities, countries, green areas and air pollution.

Entities:
- schema:City
  - schema:name (city label)
  - schema:identifier (city code)
  - schema:containedInPlace -> schema:Country

- schema:Country
  - schema:name (country label)

Measurements are modeled as sosa:Observation:
- sosa:hasFeatureOfInterest links an observation to a city
- sosa:hasSimpleResult contains the numeric value
- schema:qualitativeValue contains the category label (for pollution observations)
- sosa:resultTime contains the year (for green observations: "2020"^^xsd:gYear)

Observed properties (via sosa:observedProperty):
- PM2.5: <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/6001>
- O3:    <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/7>
- NO2:   <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/8>
- CO:    <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/10>
- AQI:   <http://dbpedia.org/resource/Air_quality_index>

Green indicators are observations identified by schema:name:
- "Green Area Per Capita" (m2/person)
- "Green Area Share" (percent)

IMPORTANT: All measurements use sosa:Observation pattern, NOT direct properties on cities.
"""

schema = GraphSchema()
print("Graph schema defined")
print(schema.prefixes)

Graph schema defined

PREFIX schema: <http://schema.org/>
PREFIX sosa:   <http://www.w3.org/ns/sosa/>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>



In [25]:
class OllamaClient:
    """Client for communicating with local Ollama instance."""

    def __init__(self, model: str = "llama2", base_url: str = "http://localhost:11434"):
        self.model = model
        self.base_url = base_url
        self.api_url = f"{base_url}/api/generate"

    def is_available(self) -> bool:
        """Check if Ollama server is reachable."""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=2)
            return response.status_code == 200
        except:
            return False

    def list_models(self) -> List[str]:
        """Return list of available models."""
        try:
            response = requests.get(f"{self.base_url}/api/tags")
            if response.status_code == 200:
                data = response.json()
                return [model['name'] for model in data.get('models', [])]
            return []
        except:
            return []

    def generate(self, prompt: str, temperature: float = 0.3) -> str:
        """Generate a response from the model."""
        if OLLAMA_PACKAGE_AVAILABLE:
            try:
                response = ollama.generate(model=self.model, prompt=prompt)
                return response['response']
            except Exception as e:
                print(f"ollama package error: {e}")
                return self._generate_with_requests(prompt, temperature)
        else:
            return self._generate_with_requests(prompt, temperature)

    def _generate_with_requests(self, prompt: str, temperature: float) -> str:
        """Fallback generation using requests."""
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": temperature}
        }
        try:
            response = requests.post(self.api_url, json=payload, timeout=60)
            if response.status_code == 200:
                return response.json()['response']
            else:
                return f"Error: {response.status_code} - {response.text}"
        except Exception as e:
            return f"Connection error: {str(e)}"

print("OllamaClient defined")

OllamaClient defined


In [26]:
client = OllamaClient(model="llama3.2:latest")

if client.is_available():
    print("Ollama is available")
    models = client.list_models()
    print(f"Available models: {models}")
    print(f"Using model: {client.model}")
else:
    print("Ollama is not available")
    print("Ensure Ollama is installed and running (ollama serve)")

Ollama is available
Available models: ['llama3.2:latest']
Using model: llama3.2:latest


In [27]:
def validate_sparql(query: str) -> Tuple[str, List[str]]:
    """
    Validate and fix common LLM errors in generated SPARQL.
    Returns (fixed_query, list_of_warnings).
    """
    warnings = []
    fixed = query

    # Fix 1: Replace https://schema.org/ with http://schema.org/
    if "https://schema.org/" in fixed:
        fixed = fixed.replace("https://schema.org/", "http://schema.org/")
        warnings.append("Fixed: https://schema.org/ -> http://schema.org/")

    # Fix 2: Warn about ex: prefix (not used in our mapping)
    if re.search(r'\bex:', fixed):
        warnings.append("Warning: 'ex:' prefix found but not used in our RDF graph")

    # Fix 3: Check for direct city properties (common LLM mistake)
    direct_props = ['ex:aqiValue', 'ex:greenShare', 'ex:greenPerCapita', 'ex:pm25Value']
    for prop in direct_props:
        if prop in fixed:
            warnings.append(f"Error: {prop} used directly on city. Use sosa:Observation pattern instead.")

    # Fix 4: Check for sosa:hasFeatureOfInterest when querying measurements
    measurement_keywords = ['pm25', 'aqi', 'green', 'pollution', 'o3', 'no2', 'co']
    has_measurement_query = any(kw in query.lower() for kw in measurement_keywords)
    has_feature_of_interest = 'sosa:hasFeatureOfInterest' in fixed

    if has_measurement_query and not has_feature_of_interest:
        warnings.append("Warning: Query involves measurements but missing sosa:hasFeatureOfInterest")

    # Fix 5: Check for sosa:Observation when querying measurements
    if has_measurement_query and 'sosa:Observation' not in fixed:
        warnings.append("Warning: Query involves measurements but missing sosa:Observation")

    return fixed, warnings

print("Query validation function defined")

Query validation function defined


In [28]:
class LLMtoSPARQL:
    """Translates natural language questions to SPARQL using Ollama."""

    def __init__(self, ollama_client: OllamaClient, use_mock: bool = False):
        self.schema = GraphSchema()
        self.ollama = ollama_client
        self.use_mock = use_mock or not ollama_client.is_available()
        if self.use_mock:
            print("Mock mode enabled (Ollama not available)")

    def create_prompt(self, question: str) -> str:
        """Create a prompt with schema context and few-shot examples."""
        prompt = f"""You are a SPARQL query generator. Convert natural language questions to valid SPARQL queries.

{self.schema.schema_description}

SPARQL prefixes to use:
{self.schema.prefixes}

Example 1 - Green area share above threshold:
Question: "Which cities have more than 10% green area?"
SPARQL:
{self.schema.prefixes}
SELECT ?city ?cityName ?share
WHERE {{
  ?city a schema:City ;
        schema:name ?cityName .
  ?obs a sosa:Observation ;
       sosa:hasFeatureOfInterest ?city ;
       schema:name "Green Area Share" ;
       sosa:hasSimpleResult ?share .
  FILTER(?share > 10)
}}
ORDER BY DESC(?share)

Example 2 - Average PM2.5 for a country:
Question: "What is the average PM2.5 in France?"
SPARQL:
{self.schema.prefixes}
SELECT (AVG(?pm25) AS ?avgPm25)
WHERE {{
  ?city schema:containedInPlace ?country .
  ?country schema:name "France" .
  ?obs a sosa:Observation ;
       sosa:hasFeatureOfInterest ?city ;
       sosa:observedProperty <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/6001> ;
       sosa:hasSimpleResult ?pm25 .
}}

Example 3 - Good air quality and high green per capita:
Question: "Find cities with good air quality and lots of green space per capita"
SPARQL:
{self.schema.prefixes}
SELECT ?cityName ?pm25Cat ?greenM2
WHERE {{
  ?city a schema:City ;
        schema:name ?cityName .
  ?pmObs a sosa:Observation ;
         sosa:hasFeatureOfInterest ?city ;
         sosa:observedProperty <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/6001> ;
         schema:qualitativeValue ?pm25Cat ;
         sosa:hasSimpleResult ?pm25 .
  ?gObs a sosa:Observation ;
        sosa:hasFeatureOfInterest ?city ;
        schema:name "Green Area Per Capita" ;
        sosa:hasSimpleResult ?greenM2 .
  FILTER(?pm25Cat = "Good" && ?greenM2 > 10)
}}
ORDER BY DESC(?greenM2)

Now translate this question:
Question: "{question}"

Respond ONLY with the SPARQL query, no explanations. Start with PREFIX.
SPARQL:
"""
        return prompt

    def translate_to_sparql(self, question: str, validate: bool = True) -> Dict[str, any]:
        """Translate a natural language question to SPARQL."""
        if self.use_mock:
            return self._mock_translation(question)

        prompt = self.create_prompt(question)
        print(f"Sending question to Ollama ({self.ollama.model})...")

        response = self.ollama.generate(prompt, temperature=0.1)
        sparql_query = self._extract_sparql(response)

        # Validate and fix common errors
        warnings = []
        if validate:
            sparql_query, warnings = validate_sparql(sparql_query)
            if warnings:
                print("Validation warnings:")
                for w in warnings:
                    print(f"  - {w}")

        return {
            "question": question,
            "sparql_query": sparql_query,
            "raw_response": response,
            "method": f"ollama_{self.ollama.model}",
            "warnings": warnings
        }

    def _extract_sparql(self, response: str) -> str:
        """Extract SPARQL query from LLM response."""
        response = response.strip()
        if response.startswith("PREFIX"):
            return response

        lines = response.split('\n')
        sparql_lines = []
        in_sparql = False

        for line in lines:
            if line.strip().startswith("PREFIX") or line.strip().startswith("SELECT"):
                in_sparql = True
            if in_sparql:
                sparql_lines.append(line)

        return '\n'.join(sparql_lines) if sparql_lines else response

    def _mock_translation(self, question: str) -> Dict[str, any]:
        """Return a mock SPARQL query for testing without Ollama."""
        query = f"""{self.schema.prefixes}
SELECT ?city ?cityName
WHERE {{
  ?city a schema:City ;
        schema:name ?cityName .
}}
LIMIT 10"""
        return {
            "question": question,
            "sparql_query": query,
            "method": "mock",
            "warnings": []
        }

print("LLMtoSPARQL defined")

LLMtoSPARQL defined


In [29]:
COMPETENCY_QUESTIONS = [
    {
        "id": 1,
        "question": "Which are the 50 cities with the highest PM2.5 AQI and their green area per capita?",
        "aspect": "PM2.5 ranking with green data"
    },
    {
        "id": 2,
        "question": "Which cities have Green Area Share below 5% and what is their PM2.5 category?",
        "aspect": "Green area filtering with pollution category"
    },
    {
        "id": 3,
        "question": "Is there a relationship between PM2.5 and Green Area Per Capita? Return city, pm25, greenM2.",
        "aspect": "Multi-dimensional correlation analysis"
    }
]

print("Competency questions:")
for cq in COMPETENCY_QUESTIONS:
    print(f"  {cq['id']}. {cq['question']}")

Competency questions:
  1. Which are the 50 cities with the highest PM2.5 AQI and their green area per capita?
  2. Which cities have Green Area Share below 5% and what is their PM2.5 category?
  3. Is there a relationship between PM2.5 and Green Area Per Capita? Return city, pm25, greenM2.


In [30]:
translator = LLMtoSPARQL(client, use_mock=False)

print("=" * 80)
print("LLM to SPARQL Translation Tests")
print("=" * 80)

LLM to SPARQL Translation Tests


In [31]:
# Question 1: PM2.5 ranking with green data
cq1 = COMPETENCY_QUESTIONS[0]
print(f"Question #{cq1['id']}: {cq1['question']}")
print(f"Aspect: {cq1['aspect']}")
print("-" * 80)

result1 = translator.translate_to_sparql(cq1['question'])

print("\nGenerated SPARQL:")
print(result1['sparql_query'])

Question #1: Which are the 50 cities with the highest PM2.5 AQI and their green area per capita?
Aspect: PM2.5 ranking with green data
--------------------------------------------------------------------------------
Sending question to Ollama (llama3.2:latest)...

Generated SPARQL:
PREFIX schema: <http://schema.org/>
PREFIX sosa:   <http://www.w3.org/ns/sosa/>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?cityName ?pm25Cat ?greenM2
WHERE {
  ?city a schema:City ;
        schema:name ?cityName .
  ?obs a sosa:Observation ;
       sosa:hasFeatureOfInterest ?city ;
       schema:name "Air Quality Index" ;
       sosa:hasSimpleResult ?aqi .
  ?pm25Obs a sosa:Observation ;
         sosa:hasFeatureOfInterest ?city ;
         sosa:observedProperty <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/6001> ;
         schema:qualitativeValue ?pm25Cat ;
         sosa:hasSimpleResult ?pm25 .
  ?gObs a sosa:Observation ;
   

In [32]:
# Question 2: Green area filtering with pollution category
cq2 = COMPETENCY_QUESTIONS[1]
print(f"Question #{cq2['id']}: {cq2['question']}")
print(f"Aspect: {cq2['aspect']}")
print("-" * 80)

result2 = translator.translate_to_sparql(cq2['question'])

print("\nGenerated SPARQL:")
print(result2['sparql_query'])

Question #2: Which cities have Green Area Share below 5% and what is their PM2.5 category?
Aspect: Green area filtering with pollution category
--------------------------------------------------------------------------------
Sending question to Ollama (llama3.2:latest)...

Generated SPARQL:
PREFIX schema: <http://schema.org/>
PREFIX sosa:   <http://www.w3.org/ns/sosa/>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?cityName (CASE ?pm25Cat WHEN "Good" THEN "Good" ELSE ?pm25Cat END) AS ?pm25Cat
WHERE {
  ?city a schema:City ;
        schema:name ?cityName .
  ?obs a sosa:Observation ;
       sosa:hasFeatureOfInterest ?city ;
       schema:name "Green Area Share" ;
       sosa:hasSimpleResult ?share ;
       sosa:hasSimpleResult ?pm25 .
  FILTER(?share < 5)
  BIND(sosa:label(?pm25) AS ?pm25Cat)
}


In [33]:
# Question 3: Correlation analysis
cq3 = COMPETENCY_QUESTIONS[2]
print(f"Question #{cq3['id']}: {cq3['question']}")
print(f"Aspect: {cq3['aspect']}")
print("-" * 80)

result3 = translator.translate_to_sparql(cq3['question'])

print("\nGenerated SPARQL:")
print(result3['sparql_query'])

Question #3: Is there a relationship between PM2.5 and Green Area Per Capita? Return city, pm25, greenM2.
Aspect: Multi-dimensional correlation analysis
--------------------------------------------------------------------------------
Sending question to Ollama (llama3.2:latest)...

Generated SPARQL:
PREFIX schema: <http://schema.org/>
PREFIX sosa:   <http://www.w3.org/ns/sosa/>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?city ?pm25 ?greenM2
WHERE {
  ?obs a sosa:Observation ;
       sosa:hasFeatureOfInterest ?city ;
       schema:name "Green Area Per Capita" ;
       sosa:hasSimpleResult ?greenM2 .
  OPTIONAL { ?obs sosa:observedProperty <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/6001> ;
         sosa:hasSimpleResult ?pm25 } .
}


In [34]:
# Display the full prompt sent to the LLM
example_prompt = translator.create_prompt("Which cities are the most polluted?")
print("=" * 80)
print("Example prompt sent to Ollama")
print("=" * 80)
print(example_prompt)

Example prompt sent to Ollama
You are a SPARQL query generator. Convert natural language questions to valid SPARQL queries.


Knowledge graph about cities, countries, green areas and air pollution.

Entities:
- schema:City
  - schema:name (city label)
  - schema:identifier (city code)
  - schema:containedInPlace -> schema:Country

- schema:Country
  - schema:name (country label)

Measurements are modeled as sosa:Observation:
- sosa:hasFeatureOfInterest links an observation to a city
- sosa:hasSimpleResult contains the numeric value
- schema:qualitativeValue contains the category label (for pollution observations)
- sosa:resultTime contains the year (for green observations: "2020"^^xsd:gYear)

Observed properties (via sosa:observedProperty):
- PM2.5: <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/6001>
- O3:    <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/7>
- NO2:   <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/8>
- CO:    <http://dd.eionet.europa.eu/vocabulary/aq/pol

In [36]:
# Test custom questions
custom_question = "Which cities have high PM2.5 pollution and more than 20% green area share?"
result = translator.translate_to_sparql(custom_question)
print(result['sparql_query'])

Sending question to Ollama (llama3.2:latest)...
PREFIX schema: <http://schema.org/>
PREFIX sosa:   <http://www.w3.org/ns/sosa/>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?city ?cityName ?pm25Cat ?share
WHERE {
  ?city a schema:City ;
        schema:name ?cityName .
  ?obs a sosa:Observation ;
       sosa:hasFeatureOfInterest ?city ;
       sosa:observedProperty <http://dd.eionet.europa.eu/vocabulary/aq/pollutant/6001> ;
       schema:qualitativeValue ?pm25Cat ;
       sosa:hasSimpleResult ?pm25 .
  ?gObs a sosa:Observation ;
        sosa:hasFeatureOfInterest ?city ;
        schema:name "Green Area Share" ;
        sosa:hasSimpleResult ?share .
  FILTER(?pm25Cat = "High" && ?share > 20)
}
ORDER BY DESC(?share)
